# 2016 BCHH infrasound relative-calibration check — reconciled-time workflow

Purpose: test whether the relative gains of the colocated BCHH infrasound channels were stable before and after the 1 September 2016 AMOS-6 explosion.

This version deliberately separates three questions: **is T0 trustworthy?**, **is there a coherent pressure signal?**, and only then **are the relative gains stable?** Raw-count ratios are never computed from the unprocessed 270-s extraction window. Counts are linearly detrended, demeaned, tapered and zero-phase high-pass filtered at 0.1 Hz first; ratios are computed only inside an inspected coherent signal window.

The hand-edited EnhancedSDSClient import, SDS/StationXML paths and BCHH DD1/DD2/DD3 channel IDs are intentionally preserved.


In [ ]:
from pathlib import Path
import sqlite3
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime, Stream, read_inventory
from flovopy.enhanced.sdsclient import EnhancedSDSClient

# Locate repository root whether Jupyter starts there or in 04_event_catalog/.
here = Path.cwd().resolve()
ROOT = next(
    (p for p in [here, *here.parents]
     if (p / "data").exists() and (p / "pyproject.toml").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Could not locate repository root")

DB = ROOT / "data/processed/ksc_rockets.sqlite"
SDS_ROOT = Path("/Volumes/classdata/remastered/SDS_KSC")
XML = Path("/Volumes/classdata/KSC/station_metadata/KSC.xml")

PRESSURE_IDS = [
    "1R.BCHH.10.DD1",
    "1R.BCHH.10.DD2",
    "1R.BCHH.10.DD3",
]

for label, path in {"Database": DB, "SDS": SDS_ROOT, "StationXML": XML}.items():
    print(f"{label}: {path}")
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")

sds = EnhancedSDSClient(str(SDS_ROOT))
inv = read_inventory(str(XML))
print("Pressure channels:", PRESSURE_IDS)


## 1. Query reconciled event times and provenance

For ordinary launches used in the calibration comparison, prefer events supported by at least two **actual-event-time** sources. Scheduled launch times are provenance only and are not treated as physical waveform events.


In [ ]:
START = "2016-07-01T00:00:00Z"
END   = "2016-10-01T00:00:00Z"
RECON = ROOT / "data/processed/event_reconciliation.csv"
SOURCES = ROOT / "data/processed/event_time_sources.csv"

if RECON.exists():
    all_events = pd.read_csv(RECON, dtype=str).fillna("")
    all_events["event_time"] = pd.to_datetime(all_events["event_time_utc"], format="ISO8601", utc=True)
    all_events["source_count"] = pd.to_numeric(all_events.get("source_count", 0), errors="coerce").fillna(0).astype(int)
    all_events["time_spread_s"] = pd.to_numeric(all_events.get("time_spread_s", np.nan), errors="coerce")
    events = all_events[(all_events.event_time >= pd.Timestamp(START)) & (all_events.event_time < pd.Timestamp(END))].copy()
else:
    with sqlite3.connect(DB) as con:
        events = pd.read_sql_query("SELECT event_id,event_type,event_time_utc,mission,vehicle,pad,notes FROM events WHERE event_time_utc >= ? AND event_time_utc < ? ORDER BY event_time_utc", con, params=(START,END))
    events["event_time"] = pd.to_datetime(events["event_time_utc"], format="ISO8601", utc=True)
    events["source_count"] = 0
    events["time_spread_s"] = np.nan

# Show provenance for the events we may analyze.  Two agreeing sources is strong enough
# for the ordinary launches used in this calibration test.
if SOURCES.exists():
    source_times = pd.read_csv(SOURCES, dtype=str).fillna("")
    source_times = source_times[source_times.event_id.isin(events.event_id)]
    display(source_times[[c for c in ["event_id","source_name","time_utc","precision","time_definition","time_role","source_name_value"] if c in source_times.columns]].sort_values(["event_id","time_utc"]))

cols=[c for c in ["event_id","event_type","event_time_utc","mission","pad","source_count","time_spread_s","time_conflict"] if c in events.columns]
display(events[cols])


**T0 acceptance for the calibration comparison:** JCSAT-16, AFSPC-6 and OSIRIS-REx should each have at least two actual-time sources with negligible spread. AMOS-6 is treated separately as an extreme-amplitude survival/nonlinearity test. If this provenance table does not show that, stop before interpreting calibration ratios.


In [ ]:
# Show the independent source times for the core calibration events.
core_mask = events["mission"].astype(str).str.contains(
    "JCSAT|AFSPC|OSIRIS|AMOS", case=False, regex=True, na=False
)
core_ids = set(events.loc[core_mask, "event_id"])
time_sources[time_sources.event_id.isin(core_ids)]


## 2. Read the three BCHH pressure channels

Use explicit trace IDs rather than wildcard discovery. The broad extraction window is only for diagnostics and signal finding.


In [ ]:
PRE_S = 30
POST_S = 240

def read_event(row):
    t0 = UTCDateTime(row["event_time_utc"])
    return sds.read(
        starttime=t0-PRE_S,
        endtime=t0+POST_S,
        trace_ids=PRESSURE_IDS,
        skip_low_rate_channels=False,
        merge=None, trim=True, daywise=False,
        postprocess=False, final_smart_merge=False, verbose=False,
    )

event_streams = {}
availability_rows = []
for _, row in events.iterrows():
    try:
        st = read_event(row)
        event_streams[row.event_id] = st
        print(row.event_time_utc, row.mission, sorted({tr.id for tr in st}))
        for tr in st:
            availability_rows.append({
                "event_id": row.event_id, "mission": row.mission, "seed_id": tr.id,
                "sampling_rate": float(tr.stats.sampling_rate), "npts": int(tr.stats.npts),
                "starttime": str(tr.stats.starttime), "endtime": str(tr.stats.endtime),
            })
    except Exception as exc:
        print(row.event_time_utc, row.mission, "READ FAILED:", exc)

availability = pd.DataFrame(availability_rows)
availability


## 3. Preprocess raw counts consistently

Every raw-count trace gets the same sequence: **linear detrend → demean → cosine taper → 0.1-Hz high-pass**. The original streams remain unchanged.


In [ ]:
HIGHPASS_HZ = 0.1
FILTER_CORNERS = 4
ZEROPHASE = True
TAPER_FRACTION = 0.05

def preprocess_counts(st):
    out = Stream()
    for tr in st:
        if tr.id not in PRESSURE_IDS:
            continue
        x = tr.copy()
        x.detrend("linear")
        x.detrend("demean")
        x.taper(max_percentage=TAPER_FRACTION, type="cosine")
        x.filter("highpass", freq=HIGHPASS_HZ, corners=FILTER_CORNERS, zerophase=ZEROPHASE)
        x.stats.units = "counts (detrended, HP 0.1 Hz)"
        out.append(x)
    return out

filtered_streams = {eid: preprocess_counts(st) for eid, st in event_streams.items()}


In [ ]:
def plot_event_stream(event_id, streams=filtered_streams, xlim=(-30, 240)):
    row = events.set_index("event_id").loc[event_id]
    t0 = UTCDateTime(row.event_time_utc)
    st = streams[event_id]
    fig, axes = plt.subplots(len(PRESSURE_IDS), 1, figsize=(12, 7), sharex=True)
    for ax, seed_id in zip(axes, PRESSURE_IDS):
        matches = [tr for tr in st if tr.id == seed_id]
        if not matches:
            ax.text(0.5, 0.5, f"{seed_id}: missing", transform=ax.transAxes, ha="center")
            continue
        tr = matches[0]
        tt = np.arange(tr.stats.npts)/tr.stats.sampling_rate + float(tr.stats.starttime-t0)
        ax.plot(tt, tr.data, linewidth=0.7)
        ax.axvline(0, linestyle="--", linewidth=1)
        ax.set_ylabel(seed_id.split(".")[-1])
        ax.set_xlim(*xlim)
    axes[-1].set_xlabel("Seconds relative to reconciled event time")
    fig.suptitle(f"{row.mission} — {row.event_time_utc}\n0.1-Hz high-pass filtered counts")
    fig.tight_layout()
    plt.show()

for eid in events.loc[core_mask, "event_id"]:
    if eid in filtered_streams:
        plot_event_stream(eid)


## 4. Find a coherent signal window

The detector requires both elevated amplitude relative to a pre-event noise window and positive inter-channel correlation. It is intentionally conservative. **Always inspect the plots.** `MANUAL_WINDOWS` can override the automatic result for any event after visual review.


In [ ]:
NOISE_WINDOW = (-25.0, -5.0)
SEARCH_WINDOW = (5.0, 200.0)
CORR_WINDOW_S = 5.0
CORR_THRESHOLD = 0.60
SNR_THRESHOLD = 3.0
SIGNAL_PAD_S = 3.0

# After inspecting plots, add event-specific overrides here, e.g.
# MANUAL_WINDOWS = {"ksc-...": (42.0, 105.0)}
MANUAL_WINDOWS = {}

def trace_on_relative_grid(tr, t0):
    t = np.arange(tr.stats.npts)/tr.stats.sampling_rate + float(tr.stats.starttime-t0)
    return t, np.asarray(tr.data, dtype=float)

def detect_signal_window(st, t0):
    traces = {tr.id: tr for tr in st if tr.id in PRESSURE_IDS}
    if len(traces) < 2:
        return None, {"reason":"fewer than two channels"}
    sr = min(float(tr.stats.sampling_rate) for tr in traces.values())
    start = max(float(tr.stats.starttime-t0) for tr in traces.values())
    end = min(float(tr.stats.endtime-t0) for tr in traces.values())
    grid = np.arange(start, end, 1/sr)
    arrays = {}
    for seed_id,tr in traces.items():
        tt,xx = trace_on_relative_grid(tr,t0)
        arrays[seed_id] = np.interp(grid,tt,xx)
    noise = (grid>=NOISE_WINDOW[0]) & (grid<=NOISE_WINDOW[1])
    if noise.sum() < 10:
        return None, {"reason":"insufficient pre-event noise"}
    noise_rms = np.median([np.sqrt(np.mean(x[noise]**2)) for x in arrays.values()])
    nwin=max(5,int(round(CORR_WINDOW_S*sr)))
    centers=[]; corrs=[]; snrs=[]
    ids=list(arrays)
    search_idx=np.where((grid>=SEARCH_WINDOW[0]) & (grid<=SEARCH_WINDOW[1]))[0]
    for k in search_idx[::max(1,nwin//5)]:
        i0=max(0,k-nwin//2); i1=min(len(grid),i0+nwin)
        if i1-i0 < nwin//2: continue
        paircorr=[]
        for a,b in combinations(ids,2):
            x=arrays[a][i0:i1]; y=arrays[b][i0:i1]
            if np.std(x)>0 and np.std(y)>0: paircorr.append(np.corrcoef(x,y)[0,1])
        rms=np.median([np.sqrt(np.mean(x[i0:i1]**2)) for x in arrays.values()])
        centers.append(grid[k]); corrs.append(np.median(paircorr) if paircorr else np.nan)
        snrs.append(rms/noise_rms if noise_rms>0 else np.nan)
    d=pd.DataFrame({"t":centers,"corr":corrs,"snr":snrs})
    good=d[(d["corr"]>=CORR_THRESHOLD) & (d["snr"]>=SNR_THRESHOLD)]
    if good.empty:
        return None, {"reason":"no coherent elevated-amplitude interval", "max_corr":d["corr"].max(), "max_snr":d["snr"].max()}
    # Use the contiguous good run containing the highest SNR point.
    step=np.median(np.diff(d.t)) if len(d)>1 else CORR_WINDOW_S
    groups=(good.t.diff().fillna(0) > 1.6*step).cumsum()
    best_group=good.loc[good.snr.idxmax()]
    gid=groups.loc[best_group.name]
    run=good[groups==gid]
    win=(max(SEARCH_WINDOW[0],run.t.min()-CORR_WINDOW_S/2-SIGNAL_PAD_S),
         min(SEARCH_WINDOW[1],run.t.max()+CORR_WINDOW_S/2+SIGNAL_PAD_S))
    return win, {"max_corr":d["corr"].max(),"max_snr":d["snr"].max(),"diagnostic":d}

signal_rows=[]; signal_windows={}
lookup=events.set_index("event_id")
for eid,st in filtered_streams.items():
    row=lookup.loc[eid]; t0=UTCDateTime(row.event_time_utc)
    auto,diag=detect_signal_window(st,t0)
    win=MANUAL_WINDOWS.get(eid,auto)
    if win is not None: signal_windows[eid]=win
    signal_rows.append({
        "event_id":eid,"event_time_utc":row.event_time_utc,"mission":row.mission,
        "event_type":row.event_type,"source_count":row.source_count,
        "time_spread_s":row.time_spread_s,"signal_found":win is not None,
        "signal_start_s":win[0] if win else np.nan,"signal_end_s":win[1] if win else np.nan,
        "max_corr":diag.get("max_corr",np.nan),"max_snr":diag.get("max_snr",np.nan),
        "reason":diag.get("reason",""),"window_source":"manual" if eid in MANUAL_WINDOWS else "automatic",
    })
signal_summary=pd.DataFrame(signal_rows)
signal_summary


In [ ]:
def plot_signal_diagnostic(event_id):
    row=lookup.loc[event_id]; t0=UTCDateTime(row.event_time_utc)
    win=signal_windows.get(event_id)
    st=filtered_streams[event_id]
    fig,ax=plt.subplots(figsize=(12,4.5))
    scales=[]
    for tr in st:
        if tr.id not in PRESSURE_IDS: continue
        tt,xx=trace_on_relative_grid(tr,t0)
        scale=np.percentile(np.abs(xx),99) or 1.0
        ax.plot(tt,xx/scale,label=tr.id,linewidth=0.7)
    ax.axvspan(*NOISE_WINDOW,alpha=0.12,label="noise window")
    if win: ax.axvspan(*win,alpha=0.18,label="selected signal")
    ax.axvline(0,linestyle="--",linewidth=1,label="event time")
    ax.set_xlim(-30,210); ax.set_xlabel("Seconds relative to event time"); ax.set_ylabel("Normalized filtered counts")
    ax.set_title(f"{row.mission} — signal-window QC")
    ax.legend(fontsize="small",ncol=2); plt.show()

for eid in events.loc[core_mask,"event_id"]:
    if eid in filtered_streams: plot_signal_diagnostic(eid)


## 5. Pairwise relative gains inside the selected signal only

These are now calculated from detrended/high-pass-filtered counts and only over the selected coherent signal interval.


In [ ]:
def trim_relative(tr,t0,win):
    return tr.copy().trim(t0+win[0],t0+win[1],pad=False)

def common_arrays(tr1,tr2):
    sr1=float(tr1.stats.sampling_rate); sr2=float(tr2.stats.sampling_rate)
    if not np.isclose(sr1,sr2,rtol=0,atol=1e-8): raise ValueError(f"Sampling rates differ: {sr1} vs {sr2}")
    sr=sr1; t0=max(tr1.stats.starttime,tr2.stats.starttime); t1=min(tr1.stats.endtime,tr2.stats.endtime)
    if t1<=t0:return np.array([]),np.array([])
    i1=int(round((t0-tr1.stats.starttime)*sr)); i2=int(round((t0-tr2.stats.starttime)*sr))
    n=min(int(np.floor((t1-t0)*sr))+1,tr1.stats.npts-i1,tr2.stats.npts-i2)
    x=np.asarray(tr1.data[i1:i1+n],float); y=np.asarray(tr2.data[i2:i2+n],float)
    good=np.isfinite(x)&np.isfinite(y); return x[good],y[good]

def pair_metrics(tr1,tr2):
    x,y=common_arrays(tr1,tr2)
    if len(x)<10:return {}
    rms=lambda z:np.sqrt(np.mean(z*z)); p99=lambda z:np.percentile(np.abs(z),99)
    denom=np.dot(y,y); slope=np.dot(y,x)/denom if denom>0 else np.nan
    return {"slope":slope,"rms_ratio":rms(x)/rms(y) if rms(y)>0 else np.nan,
            "p99_ratio":p99(x)/p99(y) if p99(y)>0 else np.nan,
            "peak_ratio":np.max(np.abs(x))/np.max(np.abs(y)) if np.max(np.abs(y))>0 else np.nan,
            "correlation":np.corrcoef(x,y)[0,1] if np.std(x)>0 and np.std(y)>0 else np.nan,"npts":len(x)}

def compute_pair_table(streams,domain):
    rows=[]
    for eid,win in signal_windows.items():
        if eid not in streams:continue
        row=lookup.loc[eid]; t0=UTCDateTime(row.event_time_utc)
        traces={tr.id:trim_relative(tr,t0,win) for tr in streams[eid] if tr.id in PRESSURE_IDS}
        for id1,id2 in combinations(PRESSURE_IDS,2):
            if id1 not in traces or id2 not in traces:continue
            m=pair_metrics(traces[id1],traces[id2])
            if m: rows.append({"event_id":eid,"event_time_utc":row.event_time_utc,"mission":row.mission,
                "event_type":row.event_type,"pair":f"{id1} / {id2}","domain":domain,
                "signal_start_s":win[0],"signal_end_s":win[1],**m})
    df=pd.DataFrame(rows)
    if not df.empty: df["event_time"]=pd.to_datetime(df.event_time_utc,format="ISO8601",utc=True)
    return df

raw_relative=compute_pair_table(filtered_streams,"filtered_raw_counts")
raw_relative


## 6. Response-correct copies to Pa

Response correction is performed on copies. The pre-filter begins at 0.05 Hz and reaches unity by 0.1 Hz; the corrected traces are then explicitly high-pass filtered at the same 0.1-Hz corner as the counts branch.


In [ ]:
PRE_FILT=(0.05,0.1,40.0,45.0)
WATER_LEVEL=60
corrected_streams={}; response_rows=[]
for eid,st in event_streams.items():
    corrected=Stream()
    for tr in st:
        if tr.id not in PRESSURE_IDS:continue
        x=tr.copy()
        try:
            x.detrend("linear"); x.detrend("demean"); x.taper(max_percentage=TAPER_FRACTION,type="cosine")
            x.remove_response(inventory=inv,output="DEF",pre_filt=PRE_FILT,water_level=WATER_LEVEL,zero_mean=False,taper=False)
            x.filter("highpass",freq=HIGHPASS_HZ,corners=FILTER_CORNERS,zerophase=ZEROPHASE)
            x.stats.units="Pa"; corrected.append(x)
            response_rows.append({"event_id":eid,"seed_id":tr.id,"response_ok":True,"error":None})
        except Exception as exc:
            response_rows.append({"event_id":eid,"seed_id":tr.id,"response_ok":False,"error":str(exc)})
    corrected_streams[eid]=corrected
response_status=pd.DataFrame(response_rows)
response_status


In [ ]:
pa_relative=compute_pair_table(corrected_streams,"response_corrected_Pa")
pa_relative


In [ ]:
AMOS6_TIME = pd.to_datetime(events.loc[events.event_type.eq("pad_explosion"), "event_time_utc"].iloc[0], utc=True) if (events.event_type == "pad_explosion").any() else pd.Timestamp("2016-09-01T13:07:15Z")
def plot_metric(df,metric,title_prefix):
    if df.empty:return
    fig,ax=plt.subplots(figsize=(10,4.5))
    for pair,g in df.groupby("pair"):
        g=g.sort_values("event_time"); ax.plot(g.event_time,g[metric],marker="o",label=pair)
    ax.axvline(AMOS6_TIME,linestyle="--",label="AMOS-6")
    ax.set_ylabel(metric); ax.set_xlabel("UTC"); ax.set_title(f"{title_prefix}: {metric}"); ax.legend(fontsize="small")
    fig.autofmt_xdate(); plt.show()
for metric in ["slope","p99_ratio","rms_ratio","peak_ratio","correlation"]:
    plot_metric(raw_relative,metric,"BCHH filtered-count relative calibration")
    plot_metric(pa_relative,metric,"BCHH response-corrected relative calibration")


## 7. Interpretation / acceptance criteria

For the calibration conclusion, concentrate first on **JCSAT-16, AFSPC-6 and OSIRIS-REx**, whose launch times should have independent agreement in the reconciled catalog. Treat AMOS-6 separately as an extreme-amplitude survival/nonlinearity test.

Before accepting a calibration step:

- visually confirm the selected signal window on every event;
- require high inter-channel correlation in that window;
- inspect clipping, timing offsets and polarity;
- identify the sensor serial number behind each NSLC for each epoch;
- compare filtered-count ratios with response-corrected Pa ratios;
- if AMOS-6 is nonlinear or clipped, do not use its strongest interval to estimate gain;
- write any accepted calibration/QC conclusion back to channel-epoch metadata rather than hiding a correction in this notebook.


In [ ]:
# Compact QC/result exports for comparison across notebook runs
OUT = ROOT / "outputs/calibration_2016"
OUT.mkdir(parents=True, exist_ok=True)
if 'signal_qc' in globals() and isinstance(signal_qc, pd.DataFrame): signal_qc.to_csv(OUT / "signal_window_qc.csv", index=False)
if 'raw_relative' in globals(): raw_relative.to_csv(OUT / "relative_gain_filtered_counts.csv", index=False)
if 'pa_relative' in globals(): pa_relative.to_csv(OUT / "relative_gain_response_corrected_pa.csv", index=False)
if 'response_status' in globals(): response_status.to_csv(OUT / "response_correction_qc.csv", index=False)
print(f"Calibration outputs: {OUT}")
